# Fire Prediction Data Ingestion Pipeline

This notebook implements the data ingestion pipeline for the fire prediction project. It collects and stores data from multiple sources:

1. **MODIS Fire Detection Data** - Fire detection events (target variables)
2. **NASA POWER Weather Data** - Humidity, temperature, precipitation, wind (weather features)
3. **NASA DEM Terrain Data** - Elevation, slope, ruggedness, curvature, canyons (terrain features)
4. **Google Earth Engine Data** - Water distance, vegetation types, fuel layers (geospatial features)

## Data Store Structure

For a sample dataset (≤10GB), we'll create the following data stores:
- `data/raw/fire_detections.parquet` - Raw MODIS fire detection data
- `data/processed/weather_features.parquet` - Weather features from NASA POWER
- `data/processed/terrain_features.parquet` - Terrain features from NASA DEM
- `data/processed/geospatial_features.parquet` - Water distance and vegetation features from GEE
- `data/processed/combined_features.parquet` - Combined feature set ready for ML

## Configuration Questions

Before running the ingestion, please consider:
1. **Sample Size**: How many fire detections should we process? (Current dataset has ~20k detections)
2. **Time Range**: What date range should we use for weather data? (14 days before fire for fuel conditioning)
3. **Geographic Scope**: Process all locations or focus on a specific region?
4. **Data Store Format**: Parquet (recommended for 10GB), HDF5, or CSV?
5. **API Rate Limits**: Should we add delays between API calls to respect rate limits?


In [1]:
# Import necessary libraries
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import geopandas as gpd
from datetime import datetime, timedelta
import os
import warnings
from pathlib import Path
import time
from typing import Dict, List, Optional, Tuple

# Data ingestion modules
import sys
sys.path.append('.')

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set up paths
DATA_DIR = Path('data')
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'

# Create directories if they don't exist
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Directories created")
print(f"  - Raw data: {RAW_DIR}")
print(f"  - Processed data: {PROCESSED_DIR}")


✓ Directories created
  - Raw data: data\raw
  - Processed data: data\processed


## Configuration

Set your configuration parameters here:


In [2]:
# ============================================================================
# CONFIGURATION - Adjust these parameters as needed
# ============================================================================

# Sample size: Number of fire detections to process (None = process all)
SAMPLE_SIZE = 3000  # Start with 100 for testing, increase for full dataset

# Date range for weather data: Days before fire detection to fetch
DAYS_BEFORE_FIRE = 14  # For 14-day fuel conditioning index

# Geographic filtering: Process all locations or filter by bounds?
# Set to None to process all locations, or specify [min_lon, min_lat, max_lon, max_lat]
GEOGRAPHIC_BOUNDS = None  # Example: [-180, -90, 180, 90] for global

# Data store format: 'parquet' (recommended), 'hdf5', or 'csv'
DATA_STORE_FORMAT = 'parquet'

# ============================================================================
# SMART RATE LIMITING FOR NASA POWER API
# ============================================================================
# NASA POWER API limits:
#   - Without API key (DEMO_KEY): 30 requests/hour, 50/day
#   - With personal API key: 1,000 requests/hour (FREE!)
# Get a FREE API key at: https://api.nasa.gov/
# Set environment variable: NASA_POWER_API_KEY=your_key_here

import os
NASA_API_KEY = os.getenv('NASA_POWER_API_KEY')

if NASA_API_KEY:
    # With API key: 1000 req/hr = ~3.6 sec minimum between requests
    # Use adaptive delay with concurrent requests for speed
    API_DELAY = 0.1  # Short delay between batches
    RATE_LIMIT_DELAY = 3.6  # Delay to respect 1000/hr limit
    MAX_CONCURRENT = 5  # Number of concurrent requests
    REQUESTS_PER_HOUR = 1000
    print("✓ NASA POWER API key detected - using optimized rate limits")
    print(f"  Rate: Up to {REQUESTS_PER_HOUR} requests/hour")
else:
    # Without API key: 30 req/hr = 120 sec between requests!
    API_DELAY = 120  # 2 minutes between requests (30/hr limit)
    RATE_LIMIT_DELAY = 120
    MAX_CONCURRENT = 1  # No concurrent requests allowed
    REQUESTS_PER_HOUR = 30
    print("⚠️  No NASA POWER API key found!")
    print("   Using DEMO_KEY: limited to 30 requests/hour (2 min delay)")
    print("   ⏱️  Estimated time for 3000 samples: ~100 hours!")
    print("")
    print("   🚀 GET FREE API KEY (5 min setup):")
    print("   1. Go to: https://api.nasa.gov/")
    print("   2. Sign up for free API key")
    print("   3. Set environment variable: NASA_POWER_API_KEY=your_key")
    print("   4. Re-run this notebook")
    print("")
    print("   With API key: 3000 samples in ~3 hours (33x faster!)")

# Batch processing: Process features in batches
BATCH_SIZE = 10  # Process 10 fire detections at a time

# Feature flags: Which features to fetch
FETCH_WEATHER = True
FETCH_TERRAIN = True
FETCH_WATER_DISTANCE = True
FETCH_VEGETATION = True

print("Configuration:")
print(f"  Sample size: {SAMPLE_SIZE if SAMPLE_SIZE else 'All'}")
print(f"  Days before fire: {DAYS_BEFORE_FIRE}")
print(f"  Geographic bounds: {GEOGRAPHIC_BOUNDS if GEOGRAPHIC_BOUNDS else 'All locations'}")
print(f"  Data format: {DATA_STORE_FORMAT}")
print(f"  API delay: {API_DELAY}s")
print(f"  Batch size: {BATCH_SIZE}")


✓ NASA POWER API key detected - using optimized rate limits
  Rate: Up to 1000 requests/hour
Configuration:
  Sample size: 3000
  Days before fire: 14
  Geographic bounds: All locations
  Data format: parquet
  API delay: 0.1s
  Batch size: 10


## Step 1: Load MODIS Fire Detection Data

Load the raw fire detection data from the shapefile.


In [3]:
# Load MODIS fire detection shapefile
shapefile_path = 'data_ingest/modis_fire/MODIS_C6_1_Global_24h.shp'

try:
    print("Loading MODIS fire detection data...")
    fire_data = gpd.read_file(shapefile_path)
    
    print(f"✓ Successfully loaded {len(fire_data)} fire detections")
    
    # Apply geographic filtering if specified
    if GEOGRAPHIC_BOUNDS:
        min_lon, min_lat, max_lon, max_lat = GEOGRAPHIC_BOUNDS
        mask = (
            (fire_data['LONGITUDE'] >= min_lon) & 
            (fire_data['LONGITUDE'] <= max_lon) &
            (fire_data['LATITUDE'] >= min_lat) & 
            (fire_data['LATITUDE'] <= max_lat)
        )
        fire_data = fire_data[mask]
        print(f"✓ Filtered to {len(fire_data)} detections within bounds")
    
    # Sample if specified
    # Stratified geographic sampling for diversity
    def stratified_geographic_sampling(fire_data, sample_size, random_state=42):
        """
        Stratified sampling to ensure geographic diversity across different regions.
        Divides the world into geographic bins and samples proportionally from each.
        """
        import numpy as np
        
        fire_data = fire_data.copy()
        # Create geographic bins (lat/lon grid)
        # Divide latitude into 6 bins (roughly 30 degrees each)
        # Divide longitude into 12 bins (roughly 30 degrees each)
        n_lat_bins = 6
        n_lon_bins = 12
        
        fire_data['lat_bin'] = pd.cut(fire_data['LATITUDE'], bins=n_lat_bins, labels=False)
        fire_data['lon_bin'] = pd.cut(fire_data['LONGITUDE'], bins=n_lon_bins, labels=False)
        fire_data['geo_bin'] = fire_data['lat_bin'].astype(str) + '_' + fire_data['lon_bin'].astype(str)
        
        # Calculate samples per bin (proportional to bin size, with minimum)
        bin_counts = fire_data['geo_bin'].value_counts()
        total_fires = len(fire_data)
        min_per_bin = max(1, sample_size // (n_lat_bins * n_lon_bins))  # At least 1 per bin
        
        sampled_data = []
        remaining_samples = sample_size
        
        # Sample from each bin proportionally
        for geo_bin in bin_counts.index:
            bin_data = fire_data[fire_data['geo_bin'] == geo_bin]
            bin_size = len(bin_data)
            
            if bin_size == 0:
                continue
                
            # Proportional sampling with minimum
            bin_proportion = bin_size / total_fires
            bin_samples = max(min_per_bin, int(bin_proportion * sample_size))
            bin_samples = min(bin_samples, bin_size, remaining_samples)
            
            if bin_samples > 0:
                sampled = bin_data.sample(n=bin_samples, random_state=random_state)
                sampled_data.append(sampled)
                remaining_samples -= bin_samples
        
        # If we haven't reached sample_size, randomly sample from remaining data
        if remaining_samples > 0:
            all_sampled = pd.concat(sampled_data, ignore_index=True) if sampled_data else pd.DataFrame()
            remaining_data = fire_data[~fire_data.index.isin(all_sampled.index)]
            if len(remaining_data) > 0:
                additional = remaining_data.sample(n=min(remaining_samples, len(remaining_data)), 
                                               random_state=random_state)
                sampled_data.append(additional)
        
        result = pd.concat(sampled_data, ignore_index=True) if sampled_data else fire_data
        result = result.drop(columns=['lat_bin', 'lon_bin', 'geo_bin'], errors='ignore')
        
        print(f"  Geographic diversity: {result['geo_bin'].nunique() if 'geo_bin' in result.columns else len(result)} unique regions")
        return result
    
    # Sample if specified - use stratified sampling for geographic diversity
    if SAMPLE_SIZE and len(fire_data) > SAMPLE_SIZE:
        print(f"Applying stratified geographic sampling to select {SAMPLE_SIZE} samples...")
        fire_data = stratified_geographic_sampling(fire_data, SAMPLE_SIZE, random_state=42)
        print(f"✓ Sampled to {len(fire_data)} detections with geographic diversity")
    else:
        print(f"✓ Using all {len(fire_data)} detections")
    
    # Ensure ACQ_DATE is datetime
    if not pd.api.types.is_datetime64_any_dtype(fire_data['ACQ_DATE']):
        fire_data['ACQ_DATE'] = pd.to_datetime(fire_data['ACQ_DATE'])
    
    # Sort by date for consistent processing
    fire_data = fire_data.sort_values('ACQ_DATE').reset_index(drop=True)
    
    print(f"\nDate range: {fire_data['ACQ_DATE'].min()} to {fire_data['ACQ_DATE'].max()}")
    print(f"Geographic bounds: {fire_data.total_bounds}")
    
    # Check for future dates (NASA POWER doesn't have future data)
    today = pd.Timestamp.now()
    future_dates = fire_data[fire_data['ACQ_DATE'] > today]
    if len(future_dates) > 0:
        print(f"\n⚠ WARNING: {len(future_dates)} fire detections have future dates (after {today.date()})")
        print(f"  NASA POWER API does not have data for future dates.")
        print(f"  Weather features will be empty for these detections.")
        print(f"  Consider using historical fire data for testing.")
    
    # Save raw fire detection data
    output_path = RAW_DIR / 'fire_detections.parquet'
    fire_data.to_parquet(output_path, index=False)
    print(f"✓ Saved raw fire detections to {output_path}")
    
except Exception as e:
    print(f"✗ Error loading fire detection data: {e}")
    raise


Loading MODIS fire detection data...
✓ Successfully loaded 20731 fire detections
Applying stratified geographic sampling to select 3000 samples...
  Geographic diversity: 3000 unique regions
✓ Sampled to 3000 detections with geographic diversity

Date range: 2026-01-05 00:00:00 to 2026-01-06 00:00:00
Geographic bounds: [-109.63081  -34.08541  139.34987   36.5514 ]
✓ Saved raw fire detections to data\raw\fire_detections.parquet


## Step 2: Fetch Weather Features (NASA POWER)

Fetch weather data for each fire detection location, including:
- Relative humidity (RH2M) - for 3-day humid index and 14-day fuel conditioning
- Precipitation (PRECTOT) - for 3-day dry/wet index and weighted extremes
- Wind speed (WS10M) - for weighted weather extremes
- Temperature (T2M) - for soft binary threshold


*Below is a simple api testing script*


In [4]:
from data_ingest.nasa_power.get_humidity import fetch_fire_prediction_weather

row = fire_data.iloc[0]
fire_date = pd.to_datetime(row['ACQ_DATE'])
start_date = (fire_date - timedelta(days=DAYS_BEFORE_FIRE)).strftime('%Y%m%d')
end_date = fire_date.strftime('%Y%m%d')

raw = fetch_fire_prediction_weather(
    latitude=row['LATITUDE'],
    longitude=row['LONGITUDE'],
    start_date=start_date,
    end_date=end_date,
    units='metric',
    return_dataframe=False  # <-- important
)
print(raw)

Error fetching data from NASA POWER API: 422 Client Error:  for url: https://power.larc.nasa.gov/api/temporal/daily/point?start=20251222&end=20260105&latitude=8.8244&longitude=-0.86251&community=ag&parameters=RH2M%2CRH2M_MAX%2CRH2M_MIN%2CPRECTOT%2CWS10M%2CWS10M_MAX%2CT2M%2CT2M_MAX%2CT2M_MIN%2CPS&format=json&units=metric&user=Drew&header=true&time-standard=lst&key=zJ9rJBeKhgb1DNXsgV847uZrgKU2PZerIdSYBTTp
Empty DataFrame
Columns: []
Index: []


In [5]:
if FETCH_WEATHER:
    try:
        from data_ingest.nasa_power.get_humidity import fetch_fire_prediction_weather_batch_sync
        import time
        from datetime import timedelta
        
        print("Fetching weather features from NASA POWER (concurrent)...")
        print(f"  Rate limit: {REQUESTS_PER_HOUR} requests/hour")
        print(f"  Concurrent requests: {MAX_CONCURRENT}")
        
        # Estimate time
        total_requests = len(fire_data)
        effective_rate = REQUESTS_PER_HOUR * MAX_CONCURRENT
        est_hours = total_requests / effective_rate
        print(f"  Total locations: {total_requests}")
        print(f"  Estimated time: {est_hours:.1f} hours ({est_hours*60:.0f} minutes)")
        print()
        
        # Prepare batch data
        locations = []
        start_dates = []
        end_dates = []
        
        for idx, row in fire_data.iterrows():
            fire_date = pd.to_datetime(row['ACQ_DATE'])
            start_date = (fire_date - timedelta(days=DAYS_BEFORE_FIRE)).strftime('%Y%m%d')
            end_date = fire_date.strftime('%Y%m%d')
            
            locations.append({
                'latitude': row['LATITUDE'],
                'longitude': row['LONGITUDE'],
                'fire_index': idx,
                'fire_date': fire_date
            })
            start_dates.append(start_date)
            end_dates.append(end_date)
        
        # Process in batches to show progress and manage memory
        batch_size = 50  # Process 50 locations at a time
        all_results = []
        start_time = time.time()
        
        for batch_start in range(0, len(locations), batch_size):
            batch_end = min(batch_start + batch_size, len(locations))
            batch_locations = locations[batch_start:batch_end]
            batch_start_dates = start_dates[batch_start:batch_end]
            batch_end_dates = end_dates[batch_start:batch_end]
            
            print(f"  Processing batch {batch_start}-{batch_end-1}...")
            
            # Fetch batch concurrently
            batch_results = fetch_fire_prediction_weather_batch_sync(
                locations=batch_locations,
                start_dates=batch_start_dates,
                end_dates=batch_end_dates,
                max_concurrent=MAX_CONCURRENT,
                units='metric',
                temporal='daily'  # Use daily data for efficiency
            )
            
            all_results.extend(batch_results)
            
            # Show progress
            elapsed = time.time() - start_time
            completed = batch_end
            if completed > 0:
                rate = completed / elapsed * 3600  # requests per hour
                remaining = total_requests - completed
                eta_hours = remaining / rate if rate > 0 else 0
                success_count = sum(1 for r in all_results if r.get('success', False))
                print(f"    Status: {completed}/{total_requests} ({rate:.0f} req/hr, {success_count} success, ETA: {eta_hours:.1f}h)")
            
            # Small delay between batches to be respectful
            if batch_end < len(locations):
                time.sleep(API_DELAY)
        
        # Process results into features
        print("\nProcessing weather data into features...")
        weather_features = []
        
        for result in all_results:
            if result.get('success', False) and result.get('data') is not None:
                df = result['data']
                loc = next((l for l in locations if l['fire_index'] == result.get('fire_index')), None)
                
                if loc and len(df) > 0:
                    weather_feature = {
                        'fire_index': result.get('fire_index'),
                        'latitude': result['latitude'],
                        'longitude': result['longitude'],
                        'fire_date': loc['fire_date'],
                        # Humidity features (use min/max from API)
                        'humidity_mean': df['RH2M'].mean() if 'RH2M' in df.columns else None,
                        'humidity_min': df['RH2M_MIN'].min() if 'RH2M_MIN' in df.columns else df['RH2M'].min() if 'RH2M' in df.columns else None,
                        'humidity_max': df['RH2M_MAX'].max() if 'RH2M_MAX' in df.columns else df['RH2M'].max() if 'RH2M' in df.columns else None,
                        # Precipitation features
                        'precip_total': df['PRECTOT'].sum() if 'PRECTOT' in df.columns else None,
                        'precip_max': df['PRECTOT'].max() if 'PRECTOT' in df.columns else None,
                        'rain_days': (df['PRECTOT'] > 0.1).sum() if 'PRECTOT' in df.columns else None,
                        # Wind features (use max from API)
                        'wind_mean': df['WS10M'].mean() if 'WS10M' in df.columns else None,
                        'wind_max': df['WS10M_MAX'].max() if 'WS10M_MAX' in df.columns else df['WS10M'].max() if 'WS10M' in df.columns else None,
                        # Temperature features (use min/max from API)
                        'temp_mean': df['T2M'].mean() if 'T2M' in df.columns else None,
                        'temp_min': df['T2M_MIN'].min() if 'T2M_MIN' in df.columns else df['T2M'].min() if 'T2M' in df.columns else None,
                        'temp_max': df['T2M_MAX'].max() if 'T2M_MAX' in df.columns else df['T2M'].max() if 'T2M' in df.columns else None,
                    }
                    weather_features.append(weather_feature)
            else:
                # Log failed fetch
                if not result.get('success', False):
                    print(f"  Warning: Failed to fetch fire {result.get('fire_index')}: {result.get('error')}")
        
        # Summary
        total_time = time.time() - start_time
        print(f"\n✓ Completed in {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")
        print(f"  Effective rate: {len(all_results)/total_time*3600:.0f} requests/hour")
        print(f"  Success rate: {len(weather_features)}/{len(all_results)} ({len(weather_features)/len(all_results)*100:.1f}%)")
        
        # Create DataFrame
        weather_df_all = pd.DataFrame(weather_features)
        
        if len(weather_df_all) > 0:
            # Save weather features
            output_path = PROCESSED_DIR / 'weather_features.parquet'
            weather_df_all.to_parquet(output_path, index=False)
            print(f"✓ Saved weather features for {len(weather_df_all)} locations to {output_path}")
        else:
            print("✗ No weather features were successfully fetched")
            
    except ImportError as e:
        print(f"✗ Error importing weather module: {e}")
        print("  Make sure data_ingest.nasa_power.get_humidity is available")
        print("  Try: pip install aiohttp nest-asyncio")
    except Exception as e:
        print(f"✗ Error fetching weather features: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⏭ Skipping weather features (FETCH_WEATHER=False)")


Fetching weather features from NASA POWER (concurrent)...
  Rate limit: 1000 requests/hour
  Concurrent requests: 5
  Total locations: 3000
  Estimated time: 0.6 hours (36 minutes)

  Processing batch 0-49...
    Status: 50/3000 (78626 req/hr, 0 success, ETA: 0.0h)
  Processing batch 50-99...
    Status: 100/3000 (76515 req/hr, 0 success, ETA: 0.0h)
  Processing batch 100-149...
    Status: 150/3000 (77391 req/hr, 0 success, ETA: 0.0h)
  Processing batch 150-199...
    Status: 200/3000 (76615 req/hr, 0 success, ETA: 0.0h)
  Processing batch 200-249...
    Status: 250/3000 (75917 req/hr, 0 success, ETA: 0.0h)
  Processing batch 250-299...
    Status: 300/3000 (74817 req/hr, 0 success, ETA: 0.0h)
  Processing batch 300-349...
    Status: 350/3000 (74929 req/hr, 0 success, ETA: 0.0h)
  Processing batch 350-399...
    Status: 400/3000 (74834 req/hr, 0 success, ETA: 0.0h)
  Processing batch 400-449...
    Status: 450/3000 (73966 req/hr, 0 success, ETA: 0.0h)
  Processing batch 450-499...
  

KeyboardInterrupt: 

## Step 2.5: Validate Humidity and Rain Data Availability

Ensure that a good portion of samples have humidity and rain data, as these are critical features.

In [ ]:
# Validate humidity and rain data availability
if FETCH_WEATHER and 'weather_df_all' in locals() and weather_df_all is not None:
    print("=" * 60)
    print("HUMIDITY AND RAIN DATA VALIDATION")
    print("=" * 60)
    
    # Check for humidity columns (RH2M or similar)
    humidity_cols = [c for c in weather_df_all.columns if 'humidity' in c.lower() or 'rh' in c.lower()]
    rain_cols = [c for c in weather_df_all.columns if 'rain' in c.lower() or 'precip' in c.lower() or 'pr' in c.lower()]
    
    print(f"\nFound humidity columns: {humidity_cols}")
    print(f"Found rain/precipitation columns: {rain_cols}")
    
    if humidity_cols:
        humidity_coverage = (weather_df_all[humidity_cols].notna().any(axis=1)).sum() / len(weather_df_all) * 100
        print(f"\nHumidity data coverage: {humidity_coverage:.1f}%")
    else:
        print("⚠️  No humidity columns found!")
        humidity_coverage = 0
    
    if rain_cols:
        rain_coverage = (weather_df_all[rain_cols].notna().any(axis=1)).sum() / len(weather_df_all) * 100
        print(f"Rain/precipitation data coverage: {rain_coverage:.1f}%")
    else:
        print("⚠️  No rain/precipitation columns found!")
        rain_coverage = 0
    
    # Target: at least 80% coverage for both
    min_coverage = 80
    if humidity_coverage < min_coverage or rain_coverage < min_coverage:
        print(f"\n⚠️  WARNING: Coverage below {min_coverage}% target!")
        print("   Consider fetching more data or checking API responses.")
    else:
        print(f"\n✓ Good coverage: Both humidity and rain data above {min_coverage}%")
    
    # Filter to ensure we keep samples with humidity/rain data
    if humidity_cols and rain_cols:
        has_humidity = weather_df_all[humidity_cols].notna().any(axis=1)
        has_rain = weather_df_all[rain_cols].notna().any(axis=1)
        has_both = has_humidity & has_rain
        
        print(f"\nSamples with both humidity and rain: {has_both.sum()} ({has_both.sum()/len(weather_df_all)*100:.1f}%)")
        
        # Keep samples that have at least humidity OR rain (prefer both)
        valid_weather_indices = weather_df_all[has_humidity | has_rain].index
        print(f"Samples with at least humidity OR rain: {len(valid_weather_indices)} ({len(valid_weather_indices)/len(weather_df_all)*100:.1f}%)")
        
        # Update fire_data to only include samples with weather data
        if len(valid_weather_indices) < len(fire_data):
            print(f"\n⚠️  Filtering fire_data from {len(fire_data)} to {len(valid_weather_indices)} samples with weather data")
            # Map weather indices back to fire_data indices
            weather_fire_indices = weather_df_all.loc[valid_weather_indices, 'fire_index'].values
            fire_data = fire_data[fire_data.index.isin(weather_fire_indices)].reset_index(drop=True)
            print(f"✓ Updated fire_data to {len(fire_data)} samples")
else:
    print("⏭ Skipping humidity/rain validation (weather data not available)")

#Step 3: Fetching Geospatial Data

In [ ]:
if FETCH_TERRAIN:
    try:
        from data_ingest.nasa_dem.get_terrain import fetch_terrain_features
        
        print("Fetching terrain features from NASA DEM...")
        terrain_features = []
        
        for idx, row in fire_data.iterrows():
            if idx % 10 == 0:
                print(f"  Processing {idx+1}/{len(fire_data)}...")
            
            try:
                # Fetch terrain features
                terrain = fetch_terrain_features(
                    latitude=row['LATITUDE'],
                    longitude=row['LONGITUDE'],
                    buffer_degrees=0.01,  # ~1km buffer
                    compute_derivatives=True
                )
                
                terrain['fire_index'] = idx
                terrain_features.append(terrain)
                
                # Rate limiting
                time.sleep(API_DELAY)
                
            except Exception as e:
                print(f"  Warning: Error processing fire {idx}: {e}")
                # Add NaN values for this location
                terrain_features.append({
                    'fire_index': idx,
                    'latitude': row['LATITUDE'],
                    'longitude': row['LONGITUDE'],
                    'elevation': np.nan,
                    'elevation_std': np.nan,
                    'slope': np.nan,
                    'slope_max': np.nan,
                    'ruggedness': np.nan,
                    'curvature': np.nan,
                    'canyons': np.nan
                })
                continue
        
        # Create DataFrame
        terrain_df = pd.DataFrame(terrain_features)
        
        if len(terrain_df) > 0:
            # Save terrain features
            output_path = PROCESSED_DIR / 'terrain_features.parquet'
            terrain_df.to_parquet(output_path, index=False)
            print(f"✓ Saved terrain features for {len(terrain_df)} locations to {output_path}")
        else:
            print("✗ No terrain features were successfully fetched")
            
    except ImportError as e:
        print(f"✗ Error importing terrain module: {e}")
        print("  Make sure data_ingest.nasa_dem.get_terrain is available")
        print("  Note: Requires NASA_DEM_API_KEY environment variable")
    except Exception as e:
        print(f"✗ Error fetching terrain features: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⏭ Skipping terrain features (FETCH_TERRAIN=False)")


✓ Using GEE credentials: C:\Users\Drewo\OneDrive\Documents\GIT\fire_prediction\fireprediction-483622-3e2ab1191a16.json
✓ Earth Engine initialized with service account: acount-1@fireprediction-483622.iam.gserviceaccount.com
Fetching terrain features from Google Earth Engine (SRTM DEM)...
  Processing 1/100...
  Processing 11/100...
  Processing 21/100...
  Processing 31/100...
  Processing 41/100...
  Processing 51/100...
  Processing 61/100...
  Processing 71/100...
  Processing 81/100...
  Processing 91/100...
✓ Saved terrain features for 100 locations to data\processed\terrain_features.parquet


## Step 4: Fetch Geospatial Features (Google Earth Engine)

Fetch geospatial features for each fire detection location:
- Distance to nearest water body
- Forest types (one-hot categorical)
- Fuel layers (one-hot categorical)


In [ ]:
if FETCH_WATER_DISTANCE or FETCH_VEGETATION:
    try:
        # ========================================================================
        # Google Earth Engine Credentials Setup
        # ========================================================================
        import os
        import json
        from pathlib import Path
        
        # Load .env file
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            pass
        
        # Get GEE_KEY (filepath to JSON)
        gee_key_path = os.getenv('GEE_KEY')
        
        if not gee_key_path:
            raise ValueError("GEE_KEY environment variable not set")
        
        # Clean up the path - remove quotes and control characters
        # Strip quotes if present
        gee_key_path = gee_key_path.strip('"\'')
        # Remove control characters (form feeds, newlines, etc.)
        gee_key_path = ''.join(c for c in gee_key_path if ord(c) >= 32 or c in '\\/')
        # Normalize path separators
        gee_key_path = gee_key_path.replace('/', '\\')
        gee_key_path = gee_key_path.strip()
        
        # If path doesn't exist, try to find the JSON file automatically
        if not os.path.isfile(gee_key_path):
            print(f"⚠ File does not exist at cleaned path, searching project directory...")
            # Try to find the JSON file in the project directory
            project_root = Path.cwd()
            json_files = list(project_root.glob('*.json'))
            if json_files:
                for jf in json_files:
                    # Check if it looks like a service account key (has client_email field)
                    try:
                        with open(jf, 'r') as f:
                            test_data = json.load(f)
                            if 'client_email' in test_data and 'private_key' in test_data:
                                print(f"✓ Found valid service account key: {jf}")
                                gee_key_path = str(jf.resolve())
                                break
                    except:
                        continue
        
        if not os.path.isfile(gee_key_path):
            raise FileNotFoundError(f"Cannot find GEE credentials file: {gee_key_path}")
        
        print(f"✓ Using GEE credentials: {gee_key_path}")
        
        # Read JSON from file
        try:
            with open(gee_key_path, 'r') as f:
                key_data = json.load(f)
            
            # Check required fields
            required = ['type', 'project_id', 'private_key_id', 'private_key', 'client_email']
            missing = [f for f in required if f not in key_data]
            if missing:
                raise ValueError(f"Missing required fields in credentials: {missing}")
                
        except json.JSONDecodeError as e:
            raise ValueError(f"Invalid JSON in credentials file: {e}")
        except Exception as e:
            raise ValueError(f"Error reading credentials file: {e}")
        
        # Initialize Earth Engine
        import ee
        
        # Create credentials using email from JSON
        service_account_email = key_data['client_email']
        credentials = ee.ServiceAccountCredentials(service_account_email, gee_key_path)
        
        # Initialize
        ee.Initialize(credentials)
        print(f"✓ Earth Engine initialized with service account: {service_account_email}")
        
        # ========================================================================
        # Import GEE modules
        # ========================================================================
        from data_ingest.google_gee.get_water_distance import get_distance_to_water
        from data_ingest.google_gee.get_vegetation_features import (
            get_forest_types, 
            get_fuel_layers
        )
        
        print("Fetching geospatial features from Google Earth Engine...")
        geospatial_features = []
        
        for idx, row in fire_data.iterrows():
            if idx % 10 == 0:
                print(f"  Processing {idx+1}/{len(fire_data)}...")
            
            try:
                fire_date = pd.to_datetime(row['ACQ_DATE'])
                fire_date_str = fire_date.strftime('%Y-%m-%d')
                
                geospatial_feature = {
                    'fire_index': idx,
                    'latitude': row['LATITUDE'],
                    'longitude': row['LONGITUDE'],
                    'fire_date': fire_date_str
                }
                
                # Fetch water distance
                if FETCH_WATER_DISTANCE:
                    try:
                        distance = get_distance_to_water(
                            latitude=row['LATITUDE'],
                            longitude=row['LONGITUDE'],
                            max_search_radius_km=50.0,  # Search within 50km
                            scale_meters=100.0,  # 100m resolution for faster processing
                            fire_date=fire_date_str
                        )
                        geospatial_feature['distance_to_water_meters'] = distance
                    except Exception as e:
                        print(f"    Warning: Error fetching water distance for fire {idx}: {e}")
                        geospatial_feature['distance_to_water_meters'] = np.nan
                
                # Fetch vegetation features
                if FETCH_VEGETATION:
                    try:
                        # Get forest types
                        forest_types = get_forest_types(
                            latitude=row['LATITUDE'],
                            longitude=row['LONGITUDE'],
                            fire_date=fire_date_str,
                            data_source='MODIS',
                            classification_scheme='IGBP',
                            scale_meters=250.0
                        )
                        # Add forest type features (excluding metadata)
                        for key, value in forest_types.items():
                            if key not in ['latitude', 'longitude', 'fire_date', 'data_source']:
                                geospatial_feature[f'forest_{key}'] = value
                        
                        # Get fuel layers
                        fuel_layers = get_fuel_layers(
                            latitude=row['LATITUDE'],
                            longitude=row['LONGITUDE'],
                            fire_date=fire_date_str,
                            scale_meters=250.0
                        )
                        # Add fuel layer features (excluding metadata)
                        for key, value in fuel_layers.items():
                            if key not in ['latitude', 'longitude', 'fire_date', 'data_source']:
                                geospatial_feature[f'fuel_{key}'] = value
                                
                    except Exception as e:
                        print(f"    Warning: Error fetching vegetation for fire {idx}: {e}")
                
                geospatial_features.append(geospatial_feature)
                
                # Rate limiting
                time.sleep(API_DELAY)
                
            except Exception as e:
                print(f"  Warning: Error processing fire {idx}: {e}")
                continue
        
        # Create DataFrame
        geospatial_df = pd.DataFrame(geospatial_features)
        
        if len(geospatial_df) > 0:
            # Save geospatial features
            output_path = PROCESSED_DIR / 'geospatial_features.parquet'
            geospatial_df.to_parquet(output_path, index=False)
            print(f"✓ Saved geospatial features for {len(geospatial_df)} locations to {output_path}")
        else:
            print("✗ No geospatial features were successfully fetched")
            
    except ImportError as e:
        print(f"✗ Error importing GEE modules: {e}")
        print("  Make sure data_ingest.google_gee modules are available")
        print("  Note: Requires SERVICE_ACCOUNT and GEE_KEY environment variables")
    except Exception as e:
        print(f"✗ Error fetching geospatial features: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⏭ Skipping geospatial features (FETCH_WATER_DISTANCE=False and FETCH_VEGETATION=False)")


✓ Using GEE credentials: C:\Users\Drewo\OneDrive\Documents\GIT\fire_prediction\fireprediction-483622-3e2ab1191a16.json
✓ Earth Engine initialized with service account: acount-1@fireprediction-483622.iam.gserviceaccount.com
Fetching geospatial features from Google Earth Engine...
  Processing 1/100...


c:\Users\Drewo\.conda\envs\fire_prediction\lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for MODIS/006/MCD12Q1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by MODIS/061/MCD12Q1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/MODIS_006_MCD12Q1

  warnings.warn(warning, category=DeprecationWarning)
c:\Users\Drewo\.conda\envs\fire_prediction\lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for MODIS/006/MOD13Q1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by MODIS/061/MOD13Q1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/MODIS_006_MOD13Q1

  warnings.warn(warning, category=DeprecationWarning)
c:\Users\Drewo\.conda\envs\fire_prediction\lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required

  Processing 11/100...
  Processing 21/100...
  Processing 31/100...
  Processing 41/100...
  Processing 51/100...
  Processing 61/100...
  Processing 71/100...
  Processing 81/100...
  Processing 91/100...
✓ Saved geospatial features for 100 locations to data\processed\geospatial_features.parquet


## Step 4.5: Validate Biome Diversity

Ensure diverse biome representation in the dataset to avoid homogeneous biome classification.

In [ ]:
# Validate biome diversity
if FETCH_VEGETATION and 'geospatial_df' in locals() and geospatial_df is not None:
    print("=" * 60)
    print("BIOME DIVERSITY VALIDATION")
    print("=" * 60)
    
    # Find vegetation/biome columns
    veg_cols = [c for c in geospatial_df.columns if any(term in c.lower() for term in ['veg', 'forest', 'biome', 'vegetation', 'grassland', 'shrub', 'savanna'])]
    
    print(f"\nFound vegetation/biome columns: {veg_cols[:10]}..." if len(veg_cols) > 10 else f"\nFound vegetation/biome columns: {veg_cols}")
    
    if veg_cols:
        # Count unique biome combinations
        biome_combinations = geospatial_df[veg_cols].apply(lambda x: '_'.join(x.astype(str)), axis=1)
        unique_biomes = biome_combinations.nunique()
        
        print(f"\nUnique biome combinations: {unique_biomes}")
        print(f"Total samples: {len(geospatial_df)}")
        
        # Check distribution
        biome_counts = biome_combinations.value_counts()
        print(f"\nTop 10 biome types:")
        for biome, count in biome_counts.head(10).items():
            print(f"  {biome[:50]}: {count} samples ({count/len(geospatial_df)*100:.1f}%)")
        
        # Check for homogeneity (if one biome dominates >50%)
        max_biome_pct = biome_counts.iloc[0] / len(geospatial_df) * 100
        
        if max_biome_pct > 50:
            print(f"\n⚠️  WARNING: Biome homogeneity detected!")
            print(f"   Most common biome represents {max_biome_pct:.1f}% of samples")
            print(f"   Target: No single biome should exceed 30-40%")
            print(f"\n   Recommendation: Consider stratified sampling by biome type")
        else:
            print(f"\n✓ Good biome diversity: No single biome exceeds 50%")
            print(f"   Most common biome: {max_biome_pct:.1f}%")
        
        # If we need more diversity, suggest resampling
        if unique_biomes < 5:
            print(f"\n⚠️  WARNING: Low biome diversity ({unique_biomes} unique biomes)")
            print(f"   Target: At least 5-10 different biome types")
            print(f"   Consider expanding geographic sampling to include more diverse regions")
        else:
            print(f"\n✓ Adequate biome diversity: {unique_biomes} unique biome types")
    else:
        print("⚠️  No vegetation/biome columns found in geospatial data!")
        print("   This may indicate an issue with vegetation feature fetching.")
else:
    print("⏭ Skipping biome validation (vegetation data not available)")

## Step 5: Combine All Features

Combine all feature sets into a single dataset ready for machine learning.


In [ ]:
print("Combining all features...")

# Start with fire detection data (base dataset)
combined_df = fire_data.copy()
combined_df['fire_index'] = combined_df.index

# Merge weather features - try from memory first, then from saved parquet
if FETCH_WEATHER:
    weather_loaded = False
    if 'weather_df_all' in locals() and weather_df_all is not None:
        try:
            weather_merge = weather_df_all.drop(columns=['latitude', 'longitude', 'fire_date'], errors='ignore')
            combined_df = combined_df.merge(weather_merge, on='fire_index', how='left')
            print(f"✓ Merged weather features from memory ({len(weather_df_all)} records)")
            weather_loaded = True
        except Exception as e:
            print(f"⚠ Error merging weather from memory: {e}")
    
    # Try loading from saved parquet file
    if not weather_loaded:
        weather_parquet = PROCESSED_DIR / 'weather_features.parquet'
        if weather_parquet.exists():
            try:
                weather_df_loaded = pd.read_parquet(weather_parquet)
                weather_merge = weather_df_loaded.drop(columns=['latitude', 'longitude', 'fire_date'], errors='ignore')
                combined_df = combined_df.merge(weather_merge, on='fire_index', how='left')
                print(f"✓ Merged weather features from parquet ({len(weather_df_loaded)} records)")
            except Exception as e:
                print(f"✗ Error loading weather parquet: {e}")
        else:
            print(f"⚠ Weather parquet not found: {weather_parquet}")

# Merge terrain features - try from memory first, then from saved parquet
if FETCH_TERRAIN:
    terrain_loaded = False
    if 'terrain_df' in locals() and terrain_df is not None:
        try:
            terrain_merge = terrain_df.drop(columns=['latitude', 'longitude'], errors='ignore')
            combined_df = combined_df.merge(terrain_merge, on='fire_index', how='left')
            print(f"✓ Merged terrain features from memory ({len(terrain_df)} records)")
            terrain_loaded = True
        except Exception as e:
            print(f"⚠ Error merging terrain from memory: {e}")
    
    # Try loading from saved parquet file
    if not terrain_loaded:
        terrain_parquet = PROCESSED_DIR / 'terrain_features.parquet'
        if terrain_parquet.exists():
            try:
                terrain_df_loaded = pd.read_parquet(terrain_parquet)
                terrain_merge = terrain_df_loaded.drop(columns=['latitude', 'longitude'], errors='ignore')
                combined_df = combined_df.merge(terrain_merge, on='fire_index', how='left')
                print(f"✓ Merged terrain features from parquet ({len(terrain_df_loaded)} records)")
            except Exception as e:
                print(f"✗ Error loading terrain parquet: {e}")
        else:
            print(f"⚠ Terrain parquet not found: {terrain_parquet}")

# Merge geospatial features - try from memory first, then from saved parquet
if FETCH_WATER_DISTANCE or FETCH_VEGETATION:
    geospatial_loaded = False
    if 'geospatial_df' in locals() and geospatial_df is not None:
        try:
            geospatial_merge = geospatial_df.drop(columns=['latitude', 'longitude', 'fire_date'], errors='ignore')
            combined_df = combined_df.merge(geospatial_merge, on='fire_index', how='left')
            print(f"✓ Merged geospatial features from memory ({len(geospatial_df)} records)")
            geospatial_loaded = True
        except Exception as e:
            print(f"⚠ Error merging geospatial from memory: {e}")
    
    # Try loading from saved parquet file
    if not geospatial_loaded:
        geospatial_parquet = PROCESSED_DIR / 'geospatial_features.parquet'
        if geospatial_parquet.exists():
            try:
                geospatial_df_loaded = pd.read_parquet(geospatial_parquet)
                geospatial_merge = geospatial_df_loaded.drop(columns=['latitude', 'longitude', 'fire_date'], errors='ignore')
                combined_df = combined_df.merge(geospatial_merge, on='fire_index', how='left')
                print(f"✓ Merged geospatial features from parquet ({len(geospatial_df_loaded)} records)")
            except Exception as e:
                print(f"✗ Error loading geospatial parquet: {e}")
        else:
            print(f"⚠ Geospatial parquet not found: {geospatial_parquet}")

# Remove fire_index column (no longer needed)
combined_df = combined_df.drop(columns=['fire_index'], errors='ignore')

# Save combined dataset
output_path = PROCESSED_DIR / 'combined_features.parquet'
combined_df.to_parquet(output_path, index=False)
print(f"\n✓ Saved combined features dataset ({len(combined_df)} rows, {len(combined_df.columns)} columns)")
print(f"  File size: {output_path.stat().st_size / (1024**2):.2f} MB")

# Display summary
print("\n=== Dataset Summary ===")
print(f"Total rows: {len(combined_df)}")
print(f"Total columns: {len(combined_df.columns)}")
print(f"\nColumn categories:")
print(f"  - Fire detection: {len([c for c in combined_df.columns if c in ['LATITUDE', 'LONGITUDE', 'BRIGHTNESS', 'CONFIDENCE', 'FRP']])}")
print(f"  - Weather: {len([c for c in combined_df.columns if 'rh2m' in c.lower() or 'precipitation' in c.lower() or 'wind' in c.lower() or 'temperature' in c.lower()])}")
print(f"  - Terrain: {len([c for c in combined_df.columns if c in ['elevation', 'slope', 'ruggedness', 'curvature', 'canyons']])}")
print(f"  - Geospatial: {len([c for c in combined_df.columns if 'distance_to_water' in c.lower() or 'forest' in c.lower() or 'fuel' in c.lower()])}")

print(f"\n✓ Data ingestion complete!")
print(f"  Combined dataset saved to: {output_path}")


Combining all features...
✓ Merged weather features from parquet (100 records)
✓ Merged terrain features from parquet (100 records)
✓ Merged geospatial features from memory (100 records)

✓ Saved combined features dataset (100 rows, 82 columns)
  File size: 0.07 MB

=== Dataset Summary ===
Total rows: 100
Total columns: 82

Column categories:
  - Fire detection: 5
  - Weather: 10
  - Terrain: 5
  - Geospatial: 51

✓ Data ingestion complete!
  Combined dataset saved to: data\processed\combined_features.parquet


In [ ]:
# ============================================================================
# VALIDATION: Display sample data from each feature category
# ============================================================================

print("=" * 80)
print("VALIDATION: Sample Data from Each Feature Category")
print("=" * 80)

# Define column categories
fire_detection_cols = [c for c in combined_df.columns if c in [
    'LATITUDE', 'LONGITUDE', 'BRIGHTNESS', 'SCAN', 'TRACK', 
    'ACQ_DATE', 'ACQ_TIME', 'SATELLITE', 'CONFIDENCE', 'VERSION', 
    'BRIGHT_T31', 'FRP', 'DAYNIGHT', 'geometry'
]]

weather_cols = [c for c in combined_df.columns if any(x in c.lower() for x in [
    'rh2m', 'precipitation', 'wind', 'temperature', 't2m', 'prectotcorr',
    'ws2m', 'ps', 'humidity', 'rain', 'temp'
])]

terrain_cols = [c for c in combined_df.columns if c.lower() in [
    'elevation', 'elevation_std', 'slope', 'slope_max', 
    'ruggedness', 'curvature', 'canyons'
]]

geospatial_cols = [c for c in combined_df.columns if any(x in c.lower() for x in [
    'distance_to_water', 'forest', 'fuel', 'vegetation', 'ndvi', 'evi', 'npp'
])]

# 1. Fire Detection Features
print("\n" + "=" * 40)
print("1. FIRE DETECTION FEATURES")
print("=" * 40)
print(f"Columns ({len(fire_detection_cols)}): {fire_detection_cols}")
if fire_detection_cols:
    display(combined_df[fire_detection_cols].head())
else:
    print("⚠ No fire detection columns found")

# 2. Weather Features
print("\n" + "=" * 40)
print("2. WEATHER FEATURES")
print("=" * 40)
print(f"Columns ({len(weather_cols)}): {weather_cols}")
if weather_cols:
    display(combined_df[weather_cols].head())
else:
    print("⚠ No weather columns found")

# 3. Terrain Features
print("\n" + "=" * 40)
print("3. TERRAIN FEATURES")
print("=" * 40)
print(f"Columns ({len(terrain_cols)}): {terrain_cols}")
if terrain_cols:
    display(combined_df[terrain_cols].head())
else:
    print("⚠ No terrain columns found")

# 4. Geospatial Features
print("\n" + "=" * 40)
print("4. GEOSPATIAL FEATURES")
print("=" * 40)
print(f"Columns ({len(geospatial_cols)}): {geospatial_cols}")
if geospatial_cols:
    display(combined_df[geospatial_cols].head())
else:
    print("⚠ No geospatial columns found")

# Summary statistics
print("\n" + "=" * 40)
print("SUMMARY: Missing Values by Category")
print("=" * 40)
for name, cols in [
    ("Fire Detection", fire_detection_cols),
    ("Weather", weather_cols),
    ("Terrain", terrain_cols),
    ("Geospatial", geospatial_cols)
]:
    if cols:
        missing = combined_df[cols].isnull().sum().sum()
        total = len(combined_df) * len(cols)
        pct = (missing / total * 100) if total > 0 else 0
        print(f"  {name}: {missing}/{total} missing values ({pct:.1f}%)")
    else:
        print(f"  {name}: No columns")


VALIDATION: Sample Data from Each Feature Category

1. FIRE DETECTION FEATURES
Columns (14): ['LATITUDE', 'LONGITUDE', 'BRIGHTNESS', 'SCAN', 'TRACK', 'ACQ_DATE', 'ACQ_TIME', 'SATELLITE', 'CONFIDENCE', 'VERSION', 'BRIGHT_T31', 'FRP', 'DAYNIGHT', 'geometry']


,LATITUDE,LONGITUDE,BRIGHTNESS,SCAN,TRACK,ACQ_DATE,ACQ_TIME,SATELLITE,CONFIDENCE,VERSION,BRIGHT_T31,FRP,DAYNIGHT,geometry
0,8.03751,17.22174,324.88,1.40,1.17,2026-01-05,1340,A,34,6.1NRT,310.28,15.70,D,POINT (17.22174 8.03751)
1,5.93590,21.27689,319.83,1.00,1.00,2026-01-05,1337,A,72,6.1NRT,305.11,9.50,D,POINT (21.27689 5.9359)
2,5.29111,30.14096,313.53,2.62,1.55,2026-01-05,0735,T,56,6.1NRT,298.90,19.45,D,POINT (30.14096 5.29111)
3,8.92654,22.14497,341.46,1.06,1.03,2026-01-05,1340,A,89,6.1NRT,309.59,33.09,D,POINT (22.14497 8.92654)
4,15.34759,20.22583,326.58,1.02,1.01,2026-01-05,1340,A,62,6.1NRT,306.42,9.95,D,POINT (20.22583 15.34759)



2. WEATHER FEATURES
Columns (10): ['rh2m_mean', 'precipitation_total', 'wind_speed_mean', 'temperature_mean', 'rh2m_3day_mean', 'precipitation_3day_total', 'rh2m_14day_mean', 'precipitation_14day_total', 'wind_extreme_12h', 'precipitation_extreme_12h']


,rh2m_mean,precipitation_total,wind_speed_mean,temperature_mean,rh2m_3day_mean,precipitation_3day_total,rh2m_14day_mean,precipitation_14day_total,wind_extreme_12h,precipitation_extreme_12h
0,23.839222,NaN,1.852750,29.058389,23.065417,NaN,23.839222,NaN,2.38,NaN
1,42.518250,NaN,1.235417,28.283667,31.903611,NaN,42.518250,NaN,1.94,NaN
2,43.649500,NaN,1.628361,27.992444,33.835694,NaN,43.649500,NaN,2.53,NaN
3,21.867806,NaN,2.208250,28.242167,21.247639,NaN,21.867806,NaN,3.46,NaN
4,15.066111,NaN,4.185639,24.764417,15.292917,NaN,15.066111,NaN,5.53,NaN



3. TERRAIN FEATURES
Columns (7): ['elevation', 'elevation_std', 'slope', 'slope_max', 'ruggedness', 'curvature', 'canyons']


,elevation,elevation_std,slope,slope_max,ruggedness,curvature,canyons
0,404.499774,5.141390,2.039148,10.766813,4.204556,0.269794,0.0
1,512.131482,10.741735,2.938156,9.500520,5.863655,0.281707,0.0
2,637.042141,6.049330,2.709164,9.595970,6.350739,0.079243,0.0
3,536.686397,10.242034,4.140984,25.416286,5.667468,0.231215,0.0
4,409.512551,2.314409,2.955788,9.940180,4.754941,0.263471,0.0



4. GEOSPATIAL FEATURES
Columns (51): ['distance_to_water_meters', 'forest_forest_type_evergreen_needleleaf_forest', 'forest_forest_type_evergreen_broadleaf_forest', 'forest_forest_type_deciduous_needleleaf_forest', 'forest_forest_type_deciduous_broadleaf_forest', 'forest_forest_type_mixed_forests', 'forest_forest_type_closed_shrublands', 'forest_forest_type_open_shrublands', 'forest_forest_type_woody_savannas', 'forest_forest_type_savannas', 'forest_forest_type_grasslands', 'forest_forest_type_permanent_wetlands', 'forest_forest_type_croplands', 'forest_forest_type_urban_built_up', 'forest_forest_type_cropland_natural_mosaic', 'forest_forest_type_snow_ice', 'forest_forest_type_barren', 'forest_forest_type_water', 'forest_landcover_class_raw', 'fuel_fuel_load', 'fuel_fuel_model', 'fuel_canopy_height', 'fuel_canopy_cover', 'fuel_surface_fuel_load', 'fuel_crown_fuel_load', 'fuel_fuel_model_1', 'fuel_fuel_model_2', 'fuel_fuel_model_3', 'fuel_fuel_model_4', 'fuel_fuel_model_5', 'fuel_fuel_

,distance_to_water_meters,forest_forest_type_evergreen_needleleaf_forest,forest_forest_type_evergreen_broadleaf_forest,forest_forest_type_deciduous_needleleaf_forest,forest_forest_type_deciduous_broadleaf_forest,forest_forest_type_mixed_forests,forest_forest_type_closed_shrublands,forest_forest_type_open_shrublands,forest_forest_type_woody_savannas,forest_forest_type_savannas,...,fuel_fuel_load_high,fuel_canopy_height_low,fuel_canopy_height_medium,fuel_canopy_height_high,fuel_canopy_cover_sparse,fuel_canopy_cover_moderate,fuel_canopy_cover_dense,fuel_has_surface_fuel,fuel_has_crown_fuel,forest_forest_type_unclassified
0,50000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
1,50000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
2,50000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
3,50000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
4,50000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN



SUMMARY: Missing Values by Category
  Fire Detection: 0/1400 missing values (0.0%)
  Weather: 400/1000 missing values (40.0%)
  Terrain: 0/700 missing values (0.0%)
  Geospatial: 99/5100 missing values (1.9%)


## Summary and Next Steps

### Data Stores Created

1. **Raw Data**: `data/raw/fire_detections.parquet`
   - Original MODIS fire detection data

2. **Processed Features**:
   - `data/processed/weather_features.parquet` - Weather features from NASA POWER
   - `data/processed/terrain_features.parquet` - Terrain features from NASA DEM
   - `data/processed/geospatial_features.parquet` - Water distance and vegetation from GEE

3. **Combined Dataset**: `data/processed/combined_features.parquet`
   - All features merged together, ready for ML

### Follow-up Questions

1. **Sample Size**: Did you process enough fire detections? Consider increasing `SAMPLE_SIZE` for more robust training data.

2. **Feature Engineering**: The current implementation includes basic aggregations. Consider adding:
   - 3-day consecutive dry/wet/humid indices (normalized)
   - 14-day fuel conditioning index (weighted sum)
   - Weighted weather extremes with exponential weighting
   - Soft binary threshold (humidity/wetness vs temperature)

3. **Temporal Features**: Add circular encoding for:
   - Season (sin/cos of day of year)
   - Time of day (sin/cos of hour)

4. **Data Quality**: Check for:
   - Missing values
   - Outliers
   - Feature distributions

5. **API Credentials**: Ensure you have:
   - NASA POWER API key (optional, but recommended)
   - NASA DEM API key (required for terrain features)
   - Google Earth Engine credentials (SERVICE_ACCOUNT and GEE_KEY)

### Next Steps

1. **Feature Engineering**: Implement the advanced feature calculations mentioned in the README
2. **Data Validation**: Check data quality and handle missing values
3. **Exploratory Data Analysis**: Visualize feature distributions and correlations
4. **Model Training**: Use the combined dataset for ML model training


This file exists to be used for the data ingestion in this project